# **NOTEBOOK 2: Modelado — MLP con Validación Cruzada Temporal**

## **Objetivo**

En este notebook entrenamos un **Perceptrón Multicapa (MLP)** para predecir el precio de cierre de ETH/USDT con un horizonte de 7 días. El problema es de regresión multi-output: el modelo recibe una ventana de datos históricos y produce una secuencia de precios futuros de 336 pasos (7 días × 48 intervalos de 30 minutos).

El diseño experimental gira alrededor de una cuestión específica: **¿cuánta historia pasada necesita el modelo para predecir mejor?** Comparamos cuatro tamaños de ventana de entrada: 7, 14, 21 y 28 días.

## **Por qué un MLP y no un modelo más simple**

Dado que el **EDA** **confirmó la presencia de dependencias no lineales** (efecto ARCH, clustering de volatilidad),**los modelos lineales como ARIMA o regresión lineal tienen limitaciones estructurales para capturar estas relaciones.** El MLP, al ser un aproximador universal de funciones, puede en principio aprender mappings no lineales entre el histórico de features y el precio futuro. Este proyecto es el primer paso en esa dirección; arquitecturas recurrentes como LSTM o modelos con atención serían extensiones naturales.

In [19]:
# --- CELDA 1: Importaciones ---
import os
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime
from typing import Dict, List, Tuple, Optional

# ML & Deep Learning
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
# Validación cruzada temporal — librería CORE del proyecto
from tsxv.splitTrainValTest import split_train_val_test_groupKFold
from tsxv.splitTrain import split_train
# Tests estadísticos
from statsmodels.tsa.stattools import bds as bds_test
from statsmodels.tsa.stattools import acf as compute_acf

# Persistencia de modelos
import joblib
import json

warnings.filterwarnings("ignore")

# Configuración de visualización
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor": "#161b22",
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#e6edf3",
    "text.color": "#e6edf3",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "grid.color": "#21262d",
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
    "font.size": 11,
    "axes.titlesize": 13,
    "figure.dpi": 120,
})

COLORS = {
    "train": "#3fb950",
    "val": "#e3b341",
    "test": "#58a6ff",
    "pred": "#f78166",
    "residual": "#d2a8ff",
    "accent": "#bc8cff",
}

print("Librerías importadas correctamente")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Librerías importadas correctamente
Timestamp: 2026-04-24 10:08:09


In [20]:
# --- CELDA 2: Carga de datos pre-procesados ---
df = pd.read_parquet("../data/eth_features.parquet")
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Rango: {df.index.min()} → {df.index.max()}")

Dataset cargado: 41,760 filas × 14 columnas
Rango: 2026-03-26 15:04:00+00:00 → 2026-04-24 15:03:00+00:00


In [21]:
# --- CELDA 3: Configuración del experimento ---
# Se definen los hiperparámetros que se explorarán de forma sistemática.
# El objetivo central es comparar ventanas de entrada (lags) de 7, 14, 21 y 28 días,
# traducidos a minutos dado que los datos son de frecuencia 1m.

MINUTES_PER_DAY = 1440

# Ventanas de entrada en número de velas (1 vela = 1 minuto)
LAG_CONFIGS = {
    "7d":  7 * MINUTES_PER_DAY,   # 10,080 puntos
    "14d": 14 * MINUTES_PER_DAY,  # 20,160 puntos
    "21d": 21 * MINUTES_PER_DAY,  # 30,240 puntos
    "28d": 28 * MINUTES_PER_DAY,  # 40,320 puntos
}

# Horizonte de predicción: 7 días en minutos
HORIZON = 7 * MINUTES_PER_DAY    # 10,080 pasos al futuro
N_FOLDS = 5 # Número de folds para GroupKFold temporal
VAL_RATIO = 0.15 # Ratio de validación en GroupKFold temporal
TARGET_COL = "close"              # Variable objetivo: precio de cierre

# Arquitectura del MLP: capas ocultas de tamaño decreciente (embudo)
MLP_PARAMS = {
    "hidden_layer_sizes": (256, 128, 64, 32),
    "activation": "relu",
    "solver": "adam",
    "alpha": 1e-4,           # Regularización L2
    "learning_rate": "adaptive",
    "learning_rate_init": 1e-3,
    "max_iter": 500,
    "early_stopping": True,
    "validation_fraction": 0.1,
    "n_iter_no_change": 15,
    "random_state": 42,
    "verbose": False,
    "batch_size": "auto",
    "beta_1": 0.9,
    "beta_2": 0.999,
}

print("    Configuración del experimento:")
print(f"   Lags a explorar  : {list(LAG_CONFIGS.keys())}")
print(f"   Horizonte        : {HORIZON:,} min ({HORIZON // MINUTES_PER_DAY} días)")
print(f"   Folds GroupKFold : {N_FOLDS}")
print(f"   Target           : {TARGET_COL}")
print(f"   Arquitectura MLP : {MLP_PARAMS['hidden_layer_sizes']}")

    Configuración del experimento:
   Lags a explorar  : ['7d', '14d', '21d', '28d']
   Horizonte        : 10,080 min (7 días)
   Folds GroupKFold : 5
   Target           : close
   Arquitectura MLP : (256, 128, 64, 32)


## **Diseño del Experimento**

### **Arquitectura del MLP**

Se utiliza una arquitectura de **embudo decreciente**: `(256 → 128 → 64 → 32)` neuronas en cada capa oculta. Esta elección no es arbitraria: las primeras capas aprenden representaciones más generales de la ventana de tiempo, mientras que las capas finales comprimen esa información en una predicción concreta. Todas las capas usan activación **ReLU**, que **evita el problema del gradiente que desaparece** (vanishing gradient) presente en sigmoide o tanh para redes profundas.

### **Hiperparámetros clave**

- **Adam** como optimizador: combina momentum y RMSProp, siendo especialmente eficiente en problemas con datos ruidosos como series de tiempo financieras.
- **L2 regularization** (`alpha=1e-4`): penaliza pesos grandes para reducir sobreajuste.
- **Early stopping** con paciencia de **15 iteraciones sin mejora**: detiene el entrenamiento automáticamente cuando la pérdida en validación deja de bajar, evitando memorizar el conjunto de entrenamiento.
- **Learning rate adaptivo**: reduce la tasa de aprendizaje cuando el entrenamiento se estanca.

### **Validación Cruzada Temporal (TimeSeriesSplit)**

En series de tiempo **no podemos usar k-fold tradicional**, porque mezclaría observaciones futuras con pasadas, generando data leakage. En su lugar, usamos `TimeSeriesSplit` con **expanding window**: cada fold agrega más datos de entrenamiento manteniendo siempre el orden cronológico. Esto simula el proceso real de reentrenamiento de un modelo en producción.

In [ ]:
# --- CELDA 4: Feature engineering con timeseries-cv ---
# timeseries-cv (tsxv.splitTrain.split_train) crea las ventanas deslizantes.
# TimeSeriesSplit aplica K-fold temporal con expanding window:


RESAMPLE_FREQ = "30min"

INPUT_FEATURES = [
    "close",
    "log_return",
    "realized_vol_60",
    "price_range",
    "volume_zscore",
    "momentum_60",
    "momentum_240",
]

CLOSE_IDX = INPUT_FEATURES.index("close")


def prepare_and_split_dataset(
    df: pd.DataFrame,
    lag_minutes: int,
    horizon_minutes: int,
    resample_freq: str,
    features: List[str],
) -> Optional[Tuple]:
    df_resampled = df[features].resample(resample_freq).mean().dropna()

    if "h" in resample_freq:
        freq_min = int(resample_freq.replace("h", "")) * 60
    else:
        freq_min = int(resample_freq.replace("min", "").replace("T", ""))
    lag_steps     = lag_minutes     // freq_min
    horizon_steps = horizon_minutes // freq_min

    print(f"   Frecuencia: {resample_freq} | Lag steps: {lag_steps} | Horizon steps: {horizon_steps}")
    print(f"   Filas tras resample: {len(df_resampled):,}")

    feature_matrix = df_resampled[features].values.astype(np.float32)
    close_col = features.index("close")

    if len(feature_matrix) < lag_steps + horizon_steps:
        print(f"   Datos insuficientes ({len(feature_matrix)} < {lag_steps + horizon_steps})")
        return None

    # timeseries-cv: ventanas deslizantes solapadas
    X_list, y_list = split_train(feature_matrix, lag_steps, horizon_steps, 1)

    X_arr = np.array(X_list, dtype=np.float32)    # [n_samples, lag_steps, n_features]
    y_arr = np.array(y_list, dtype=np.float32)    # [n_samples, horizon_steps, n_features]

    X = X_arr.reshape(X_arr.shape[0], -1)          # [n_samples, lag_steps * n_features]
    y = y_arr[:, :, close_col]                     # [n_samples, horizon_steps]

    print(f"   Ventanas creadas: X={X.shape} | y={y.shape}")

    # TimeSeriesSplit: K-fold temporal con expanding window, sin data leakage
    tscv = TimeSeriesSplit(n_splits=N_FOLDS)
    fold_splits = []

    for fold_idx, (train_val_idx, test_idx) in enumerate(tscv.split(X)):
        n_val     = max(1, int(len(train_val_idx) * VAL_RATIO))
        train_idx = train_val_idx[:-n_val]
        val_idx   = train_val_idx[-n_val:]

        print(
            f"   Fold {fold_idx}: Train={len(train_idx):,} "
            f"| Val={len(val_idx):,} | Test={len(test_idx):,}"
        )
        fold_splits.append({
            "fold_idx": fold_idx,
            "X_train": X[train_idx], "y_train": y[train_idx],
            "X_val":   X[val_idx],   "y_val":   y[val_idx],
            "X_test":  X[test_idx],  "y_test":  y[test_idx],
        })

    return fold_splits, lag_steps, horizon_steps, df_resampled.index


print("Preparando datasets con timeseries-cv + TimeSeriesSplit...")

datasets = {}
for lag_name, lag_minutes in LAG_CONFIGS.items():
    print(f"\n Lag {lag_name} ({lag_minutes:,} min):")
    result = prepare_and_split_dataset(
        df=df,
        lag_minutes=lag_minutes,
        horizon_minutes=HORIZON,
        resample_freq=RESAMPLE_FREQ,
        features=INPUT_FEATURES,
    )
    if result is None:
        datasets[lag_name] = None
        continue
    fold_splits, lag_steps, horizon_steps, ts_index = result
    datasets[lag_name] = {
        "fold_splits":   fold_splits,
        "lag_steps":     lag_steps,
        "horizon_steps": horizon_steps,
        "timestamps":    ts_index,
    }

print("\n Todos los datasets preparados")


Preparando datasets con timeseries-cv + TimeSeriesSplit...

 Lag 7d (10,080 min):
   Frecuencia: 30min | Lag steps: 336 | Horizon steps: 336
   Filas tras resample: 1,393
   Ventanas creadas: X=(722, 2352) | y=(722, 336)
   Fold 0: Train=104 | Val=18 | Test=120
   Fold 1: Train=206 | Val=36 | Test=120
   Fold 2: Train=308 | Val=54 | Test=120
   Fold 3: Train=410 | Val=72 | Test=120
   Fold 4: Train=512 | Val=90 | Test=120

 Lag 14d (20,160 min):
   Frecuencia: 30min | Lag steps: 672 | Horizon steps: 336
   Filas tras resample: 1,393
   Ventanas creadas: X=(386, 4704) | y=(386, 336)
   Fold 0: Train=57 | Val=9 | Test=64
   Fold 1: Train=111 | Val=19 | Test=64
   Fold 2: Train=165 | Val=29 | Test=64
   Fold 3: Train=220 | Val=38 | Test=64
   Fold 4: Train=274 | Val=48 | Test=64

 Lag 21d (30,240 min):
   Frecuencia: 30min | Lag steps: 1008 | Horizon steps: 336
   Filas tras resample: 1,393
   Ventanas creadas: X=(50, 7056) | y=(50, 336)
   Fold 0: Train=9 | Val=1 | Test=8
   Fold 1: Trai

## **Observación sobre los Tamaños de Muestra**

Un resultado importante de la preparación de datos merece atención antes de entrenar. Al aumentar el tamaño de la ventana de entrada (lag), el número de ventanas disponibles decrece sustancialmente:

| Lag | Ventanas totales (X) | Muestras por fold (aprox.) |
|-----|---------------------|---------------------------|
| 7d  | 722                 | 104 – 512 (train)         |
| 14d | 386                 | 57 – 274 (train)          |
| 21d | 50                  | 9 – 36 (train)            |
| 28d | Insuficiente        | —                         |

La configuración de 21 días resulta en **muy pocas muestras por fold** para el primer experimento (9 observaciones de entrenamiento en el fold 1).**Esto es una señal de alerta: un MLP con cientos de parámetros entrenado sobre 9 muestras tiene altísima probabilidad de sobreajustar.** Los resultados deben interpretarse con esta limitación en mente, lo que revisaremos en detalle en el Notebook 3.

In [ ]:
# --- CELDA 5: Pipeline de entrenamiento usando splits de timeseries-cv ---
# Los splits (X_train, y_train, X_val, y_val, X_test, y_test) ya fueron
# generados por timeseries-cv en la Celda 4, sin data leakage temporal.


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Metricas de evaluacion sobre el primer horizonte (h=1)."""
    y_true_h1 = y_true[:, 0]
    y_pred_h1 = y_pred[:, 0]
    mask = np.abs(y_true_h1) > 1e-8   # evitar div/0 en MAPE
    mape = np.mean(np.abs((y_true_h1[mask] - y_pred_h1[mask]) / y_true_h1[mask])) * 100
    mae  = mean_absolute_error(y_true_h1, y_pred_h1)
    mse  = mean_squared_error(y_true_h1, y_pred_h1)
    return {"MAPE": mape, "MAE": mae, "RMSE": np.sqrt(mse), "MSE": mse}


def run_experiment(
    fold_splits: List[Dict],
    timestamps: pd.DatetimeIndex,
    lag_name: str,
) -> Dict:
    """
    Entrena y evalua el MLP sobre los 5 folds generados por timeseries-cv.

    Cada fold respeta el orden temporal estricto:
      - RobustScaler se ajusta SOLO sobre train (nunca sobre val/test).
      - Metricas calculadas en train, val y test de cada fold.
      - BDS test sobre residuos acumulados (h=1) para verificar i.i.d.

    Args:
        fold_splits : Lista de dicts con X/y por fold (salida de timeseries-cv).
        timestamps  : DatetimeIndex del DataFrame remuestreado.
        lag_name    : Etiqueta de configuracion (p.ej. '14d').
    """
    print(f"\n{'='*60}")
    print(f"EXPERIMENTO: Lag = {lag_name}  ({len(fold_splits)} folds)")
    print(f"{'='*60}")

    fold_results   = []
    all_residuals  = []
    best_fold_data = None
    best_rmse      = np.inf

    for fold in fold_splits:
        fold_idx = fold["fold_idx"]
        X_train, y_train = fold["X_train"], fold["y_train"]
        X_val,   y_val   = fold["X_val"],   fold["y_val"]
        X_test,  y_test  = fold["X_test"],  fold["y_test"]

        print(f"\n  Fold {fold_idx + 1}/{len(fold_splits)}:")
        print(f"     Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

        # RobustScaler: IQR en lugar de std, robusto a fat tails financieras.
        # fit SOLO sobre train para evitar data leakage en scaling.
        scaler = RobustScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc  = scaler.transform(X_test)
        X_val_sc   = scaler.transform(X_val) if len(X_val) > 0 else X_val

        # early_stopping requiere min ~10 samples en train; desactivar para folds pequeños
        mlp_params = dict(MLP_PARAMS)
        if len(X_train) < 20:
            mlp_params["early_stopping"] = False
        model = MultiOutputRegressor(MLPRegressor(**mlp_params), n_jobs=-1)
        model.fit(X_train_sc, y_train)

        y_pred_train = model.predict(X_train_sc)
        y_pred_test  = model.predict(X_test_sc)
        metrics_train = compute_metrics(y_train, y_pred_train)
        metrics_test  = compute_metrics(y_test,  y_pred_test)

        if len(X_val) > 0:
            y_pred_val  = model.predict(X_val_sc)
            metrics_val = compute_metrics(y_val, y_pred_val)
        else:
            y_pred_val  = np.array([])
            metrics_val = {"MAPE": np.nan, "MAE": np.nan, "RMSE": np.nan, "MSE": np.nan}

        print(f"     TRAIN -- RMSE: {metrics_train['RMSE']:.4f} | MAPE: {metrics_train['MAPE']:.4f}%")
        print(f"     VAL   -- RMSE: {metrics_val['RMSE']:.4f}  | MAPE: {metrics_val['MAPE']:.4f}%")
        print(f"     TEST  -- RMSE: {metrics_test['RMSE']:.4f}  | MAPE: {metrics_test['MAPE']:.4f}%")

        # Residuos h=1 acumulados para el test BDS posterior
        residuals_h1 = y_test[:, 0] - y_pred_test[:, 0]
        all_residuals.extend(residuals_h1.tolist())

        if metrics_test["RMSE"] < best_rmse:
            best_rmse = metrics_test["RMSE"]
            best_fold_data = {
                "fold":         fold_idx + 1,
                "model":        model,
                "scaler":       scaler,
                "X_train":      X_train,    "y_train":      y_train,
                "X_val":        X_val,      "y_val":        y_val,
                "X_test":       X_test,     "y_test":       y_test,
                "y_pred_train": y_pred_train,
                "y_pred_val":   y_pred_val,
                "y_pred_test":  y_pred_test,
                "timestamps":   timestamps,
            }

        fold_results.append({
            "fold": fold_idx + 1,
            "lag":  lag_name,
            "train_size": len(X_train),
            "val_size":   len(X_val),
            "test_size":  len(X_test),
            **{f"train_{k}": v for k, v in metrics_train.items()},
            **{f"val_{k}":   v for k, v in metrics_val.items()},
            **{f"test_{k}":  v for k, v in metrics_test.items()},
        })

    # Test BDS sobre residuos acumulados de todos los folds.
    # H0: residuos son i.i.d. Rechazo -> estructura temporal no capturada.
    residuals_arr = np.array(all_residuals)
    bds_results = {}
    for embed_dim in [2, 3, 4, 5]:
        try:
            stat_arr, pval_arr = bds_test(residuals_arr, max_dim=embed_dim)
            bds_stat = float(stat_arr[-1])
            bds_pval = float(pval_arr[-1])
        except Exception:
            bds_stat, bds_pval = np.nan, np.nan
        bds_results[f"dim_{embed_dim}"] = {"stat": bds_stat, "pval": bds_pval}

    print(f"\n  Test BDS sobre residuos (h=1):")
    for dim, res in bds_results.items():
        sig = (
            "*** (Rechaza i.i.d.)" if res["pval"] < 0.01
            else "** (Rechaza i.i.d.)"  if res["pval"] < 0.05
            else "No rechaza i.i.d."
        )
        print(f"     {dim}: stat={res['stat']:.4f} | p-val={res['pval']:.6f} {sig}")

    return {
        "fold_results":   pd.DataFrame(fold_results),
        "best_fold_data": best_fold_data,
        "bds_results":    bds_results,
        "all_residuals":  residuals_arr,
    }


In [15]:
# --- CELDA 6: Ejecucion del experimento completo ---
print("Iniciando experimento completo...\n")

experiment_results = {}
for lag_name, data in datasets.items():
    if data is None or not data["fold_splits"]:
        print(f"Lag {lag_name}: datos insuficientes, omitiendo.")
        continue
    experiment_results[lag_name] = run_experiment(
        fold_splits=data["fold_splits"],
        timestamps=data["timestamps"],
        lag_name=lag_name,
    )

print("\n" + "=" * 60)
print("EXPERIMENTO COMPLETO")
print("=" * 60)


Iniciando experimento completo...


EXPERIMENTO: Lag = 7d  (5 folds)

  Fold 1/5:
     Train: 104 | Val: 18 | Test: 120
     TRAIN -- RMSE: 353.5101 | MAPE: 16.2991%
     VAL   -- RMSE: 345.2509  | MAPE: 15.4706%
     TEST  -- RMSE: 507.0644  | MAPE: 21.0864%

  Fold 2/5:
     Train: 206 | Val: 36 | Test: 120
     TRAIN -- RMSE: 362.2470 | MAPE: 15.7526%
     VAL   -- RMSE: 188.2687  | MAPE: 6.8340%
     TEST  -- RMSE: 521.6007  | MAPE: 22.1146%

  Fold 3/5:
     Train: 308 | Val: 54 | Test: 120
     TRAIN -- RMSE: 141.2673 | MAPE: 5.8829%
     VAL   -- RMSE: 447.5073  | MAPE: 17.8331%
     TEST  -- RMSE: 1236.7601  | MAPE: 46.9141%

  Fold 4/5:
     Train: 410 | Val: 72 | Test: 120
     TRAIN -- RMSE: 108.7715 | MAPE: 4.3488%
     VAL   -- RMSE: 821.3618  | MAPE: 32.8885%
     TEST  -- RMSE: 1769.6979  | MAPE: 78.8694%

  Fold 5/5:
     Train: 512 | Val: 90 | Test: 120
     TRAIN -- RMSE: 27.4780 | MAPE: 0.3706%
     VAL   -- RMSE: 692.8445  | MAPE: 28.1329%
     TEST  -- RMSE: 1201.5

## **Análisis de los Resultados por Fold**

Antes de ver la tabla resumen, conviene observar el comportamiento fold a fold, que revela patrones importantes.

**Lag 7d:** El RMSE de entrenamiento cae drásticamente del fold 3 al 5 (de 141 a 27), pero el RMSE de test se mantiene o sube (llegando a 1,769 en el fold 4). **Esta brecha creciente entre train y test es una señal clásica de **sobreajuste progresivo**: a medida que el modelo ve más datos de entrenamiento, empieza a memorizarlos en lugar de generalizar.**

**Lag 14d:** Comportamiento similar pero más estable. La diferencia train–test es grande (138 vs 690 en el fold 5), pero menos extrema que en el lag de 7 días.

**Lag 21d:** Los dos primeros folds muestran algo inusual: RMSE de train cercano a 0 (0.32 y 0.27) con solo 9 y 16 muestras respectivamente. Esto es sobreajuste casi perfecto al conjunto de entrenamiento. Sin embargo, en los folds 4 y 5, el modelo parece encontrar un mejor equilibrio (RMSE train ~770, test ~110), lo que explica su mejor promedio global.

**En general, la varianza entre folds es muy alta para todas las configuraciones, lo cual es esperado dado el tamaño limitado del dataset (60 días de datos) y la naturaleza no estacionaria del mercado cripto.**

In [23]:
# --- CELDA 7: Tabla comparativa de métricas ---
# Se consolida la tabla de resultados para comparar los cuatro tamaños de lag.
# La métrica primaria de comparación es RMSE en el conjunto de test.

print("\nTABLA COMPARATIVA DE MÉTRICAS POR CONFIGURACIÓN DE LAG")
print("=" * 80)

all_fold_results = []
for lag_name, results in experiment_results.items():
    df_folds = results["fold_results"]
    # Añadir BDS del primer horizonte
    bds_pval_dim2 = results["bds_results"].get("dim_2", {}).get("pval", np.nan)
    df_folds["bds_pval_dim2"] = bds_pval_dim2
    all_fold_results.append(df_folds)

df_all_results = pd.concat(all_fold_results, ignore_index=True)

# Tabla resumen por lag (media y std sobre folds)
summary_metrics = ["test_MAPE", "test_MAE", "test_RMSE", "test_MSE", "bds_pval_dim2"]
summary_table = df_all_results.groupby("lag")[summary_metrics].agg(["mean", "std"]).round(4)
summary_table.columns = [f"{m}_{s}" for m, s in summary_table.columns]
summary_table = summary_table.reset_index()

print("\n🏆 Resumen por Tamaño de Lag (media ± std sobre 5 folds):")
print("-" * 80)
for _, row in summary_table.iterrows():
    print(f"\n  Lag: {row['lag']}")
    print(f"  MAPE  : {row['test_MAPE_mean']:.4f}% ± {row['test_MAPE_std']:.4f}%")
    print(f"  MAE   : {row['test_MAE_mean']:.4f}  ± {row['test_MAE_std']:.4f}")
    print(f"  RMSE  : {row['test_RMSE_mean']:.4f}  ± {row['test_RMSE_std']:.4f}")
    print(f"  MSE   : {row['test_MSE_mean']:.4f}  ± {row['test_MSE_std']:.4f}")
    print(f"  BDS p : {row['bds_pval_dim2_mean']:.6f}")

# Encontrar la mejor configuración
best_lag = summary_table.loc[summary_table["test_RMSE_mean"].idxmin(), "lag"]
print(f"\nMejor configuración: Lag = {best_lag} "
      f"(RMSE = {summary_table.loc[summary_table['lag']==best_lag, 'test_RMSE_mean'].values[0]:.4f})")

# Guardar tabla
os.makedirs("../results", exist_ok=True)
df_all_results.to_csv("../results/metrics_all_folds.csv", index=False)
summary_table.to_csv("../results/metrics_summary.csv", index=False)


TABLA COMPARATIVA DE MÉTRICAS POR CONFIGURACIÓN DE LAG

🏆 Resumen por Tamaño de Lag (media ± std sobre 5 folds):
--------------------------------------------------------------------------------

  Lag: 14d
  MAPE  : 34.2551% ± 11.2108%
  MAE   : 775.3166  ± 246.2756
  RMSE  : 860.2318  ± 297.0871
  MSE   : 810607.3438  ± 551621.8955
  BDS p : nan

  Lag: 21d
  MAPE  : 18.9140% ± 17.3062%
  MAE   : 441.9223  ± 402.0195
  RMSE  : 453.5574  ± 402.6968
  MSE   : 335446.0910  ± 451430.4996
  BDS p : nan

  Lag: 7d
  MAPE  : 43.6468% ± 23.7450%
  MAE   : 971.3514  ± 543.7010
  RMSE  : 1047.3343  ± 536.1345
  MSE   : 1326861.3219  ± 1180001.3169
  BDS p : nan

Mejor configuración: Lag = 21d (RMSE = 453.5574)


## **Interpretación de la Tabla Comparativa**

El lag de **21 días obtiene el menor RMSE promedio (453.56 USDT)**, pero esta métrica debe leerse con cuidado junto a su desviación estándar (±402 USDT), que es casi tan grande como la media misma. Esto indica que el modelo no es consistente entre folds: funciona bien en algunos periodos del mercado y muy mal en otros.

**Para contextualizarlo en el problema de negocio: un RMSE de 453 USDT sobre un precio que ronda los 2,000–2,900 USDT implica un error relativo del orden del 15–20%, similar al MAPE promedio reportado (18.9%). Para una estrategia de trading de largo plazo, este nivel de error puede ser aceptable como señal direccional (¿sube o baja?), pero para gestión de riesgo precisa o cálculo de VaR, sería insuficiente sin mejoras adicionales.**

El **test BDS** sobre los residuos, que analizaremos con detalle más adelante, complementará esta evaluación: si los residuos no son i.i.d., significa que el modelo dejó estructura predecible sin capturar.

In [17]:
# --- CELDA 8: Guardar el mejor modelo ---
best_lag_results = experiment_results[best_lag]
best_fold = best_lag_results["best_fold_data"]

os.makedirs("../models", exist_ok=True)

# Guardar modelo y scaler
joblib.dump(best_fold["model"], "../models/best_mlp_model.pkl")
joblib.dump(best_fold["scaler"], "../models/best_scaler.pkl")

# Guardar metadatos del modelo
model_metadata = {
    "best_lag": best_lag,
    "lag_minutes": LAG_CONFIGS[best_lag],
    "horizon_minutes": HORIZON,
    "horizon_steps": datasets[best_lag]["horizon_steps"],
    "resample_freq": RESAMPLE_FREQ,
    "input_features": INPUT_FEATURES,
    "n_features": len(INPUT_FEATURES),
    "mlp_params": str(MLP_PARAMS),
    "best_fold": best_fold["fold"],
    "timestamp": datetime.now().isoformat(),
}

with open("../models/model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

print(f"✅ Modelo guardado:")
print(f"   📁 models/best_mlp_model.pkl")
print(f"   📁 models/best_scaler.pkl")
print(f"   📁 models/model_metadata.json")
print(f"\n   Lag: {best_lag} | Fold: {best_fold['fold']}")

✅ Modelo guardado:
   📁 models/best_mlp_model.pkl
   📁 models/best_scaler.pkl
   📁 models/model_metadata.json

   Lag: 21d | Fold: 5
